<a href="https://colab.research.google.com/github/huimouqianxiao123/Peft-Qwen2.5-/blob/main/fastapi%E9%83%A8%E7%BD%B2%E6%A8%A1%E5%9E%8B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U transformers accelerate bitsandbytes huggingface_hub peft scikit-learn tqdm
!pip install -U fastapi uvicorn nest-asyncio openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 131.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 155.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 7.4 MB/s eta 0:00:00
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.67.3
    Uninstalling tqdm-4.67.3:
      Successfully uninstalled tqdm-4.67.3
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.18.0
    Uninstalling hugging

In [2]:
from google.colab import drive

drive.mount("/content/drive")

# 基座模型路径：按你的实际 Colab 路径修改。
MODEL_PATH = "/content/drive/MyDrive/models/qwen2.5-7b-instruct"

# 如果要加载 LoRA/QLoRA adapter，填 adapter 目录；不需要则保持空字符串。
ADAPTER_PATH = "/content/drive/MyDrive/models/qwen2.5-7b-intent-qlora"
MODEL_NAME = "qwen2.5-7b-instruct"

# 批量推理与限流配置：按显存大小调小或调大。
MAX_QUEUE_SIZE = 32          # 队列最多等待多少个请求，满了直接返回 503
MAX_BATCH_SIZE = 4           # 单次最多合并多少个请求进 GPU
BATCH_TIMEOUT_SECONDS = 0.03 # 为凑 batch 最多等 30ms，低延迟和吞吐之间折中
REQUEST_TIMEOUT_SECONDS = 180
MAX_INPUT_TOKENS = 2048      # 输入过长会截断，避免显存被超长 prompt 吃满

Mounted at /content/drive


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True

if ADAPTER_PATH:
    from peft import PeftModel

    model = PeftModel.from_pretrained(model, ADAPTER_PATH)

model.config.use_cache = True
model.eval()
print("模型加载完成:", MODEL_NAME)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


模型加载完成: qwen2.5-7b-instruct


In [4]:
import asyncio
import json
import queue
import threading
import time
import uuid
from collections import defaultdict
from dataclasses import dataclass
from typing import Any, Literal

from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field

app = FastAPI(title="Qwen2.5 OpenAI Compatible API")
generate_lock = threading.Lock()
request_queue = queue.Queue(maxsize=MAX_QUEUE_SIZE)


class ChatMessage(BaseModel):
    role: Literal["system", "user", "assistant"]
    content: str


class ChatCompletionRequest(BaseModel):
    model: str = MODEL_NAME
    messages: list[ChatMessage]
    temperature: float = Field(0.7, ge=0.0, le=2.0)
    top_p: float = Field(0.9, ge=0.0, le=1.0)
    top_k: int | None = Field(None, ge=1)
    max_tokens: int = Field(512, ge=1, le=4096)
    repetition_penalty: float = Field(1.0, ge=0.1)
    stop: str | list[str] | None = None
    stream: bool = False

    # OpenAI SDK 可能会传这些字段；这里先接收，当前不参与 transformers generate。
    presence_penalty: float | None = None
    frequency_penalty: float | None = None
    user: str | None = None


@dataclass
class InferenceJob:
    request: ChatCompletionRequest
    future: asyncio.Future
    loop: asyncio.AbstractEventLoop


def _apply_stop_words(text: str, stop: str | list[str] | None) -> str:
    if stop is None:
        return text
    stop_words = [stop] if isinstance(stop, str) else stop
    cut_index = None
    for word in stop_words:
        if not word:
            continue
        index = text.find(word)
        if index >= 0 and (cut_index is None or index < cut_index):
            cut_index = index
    return text if cut_index is None else text[:cut_index]


def _token_count(text: str) -> int:
    return len(tokenizer(text, add_special_tokens=False).input_ids)


def _generation_key(request: ChatCompletionRequest) -> tuple[Any, ...]:
    return (
        request.temperature,
        request.top_p,
        request.top_k,
        request.max_tokens,
        request.repetition_penalty,
    )


def _build_generate_kwargs(request: ChatCompletionRequest) -> dict[str, Any]:
    do_sample = request.temperature > 0
    generate_kwargs: dict[str, Any] = dict(
        max_new_tokens=request.max_tokens,
        do_sample=do_sample,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        repetition_penalty=request.repetition_penalty,
        use_cache=True,
    )
    if do_sample:
        generate_kwargs["temperature"] = request.temperature
        generate_kwargs["top_p"] = request.top_p
        if request.top_k is not None:
            generate_kwargs["top_k"] = request.top_k
    return generate_kwargs


def generate_batch_chat_completions(requests: list[ChatCompletionRequest]) -> list[dict[str, Any]]:
    for request in requests:
        if not request.messages:
            raise HTTPException(status_code=400, detail="messages 不能为空")

    prompt_texts = []
    for request in requests:
        messages = [message.model_dump() for message in request.messages]
        prompt_texts.append(
            tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
        )

    model_inputs = tokenizer(prompt_texts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_INPUT_TOKENS).to(model.device)
    generate_kwargs = _build_generate_kwargs(requests[0])

    with generate_lock, torch.inference_mode():
        generated_ids = model.generate(**model_inputs, **generate_kwargs)

    prompt_length = model_inputs.input_ids.shape[-1]
    decoded_outputs = tokenizer.batch_decode(
        [output_ids[prompt_length:] for output_ids in generated_ids],
        skip_special_tokens=True,
    )

    responses = []
    created = int(time.time())
    for index, request in enumerate(requests):
        content = decoded_outputs[index].strip()
        content = _apply_stop_words(content, request.stop)
        prompt_tokens = int(model_inputs.attention_mask[index].sum().item())
        completion_tokens = _token_count(content)
        completion_id = f"chatcmpl-{uuid.uuid4().hex}"

        responses.append(
            {
                "id": completion_id,
                "object": "chat.completion",
                "created": created,
                "model": request.model or MODEL_NAME,
                "choices": [
                    {
                        "index": 0,
                        "message": {"role": "assistant", "content": content},
                        "finish_reason": "stop",
                    }
                ],
                "usage": {
                    "prompt_tokens": prompt_tokens,
                    "completion_tokens": completion_tokens,
                    "total_tokens": prompt_tokens + completion_tokens,
                },
            }
        )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return responses


def generate_chat_completion(request: ChatCompletionRequest) -> dict[str, Any]:
    return generate_batch_chat_completions([request])[0]


def _resolve_job(job: InferenceJob, result: dict[str, Any] | None = None, error: Exception | None = None):
    if error is not None:
        job.loop.call_soon_threadsafe(job.future.set_exception, error)
    else:
        job.loop.call_soon_threadsafe(job.future.set_result, result)


def _take_batch(first_job: InferenceJob) -> list[InferenceJob]:
    jobs = [first_job]
    deadline = time.time() + BATCH_TIMEOUT_SECONDS
    while len(jobs) < MAX_BATCH_SIZE:
        remaining = deadline - time.time()
        if remaining <= 0:
            break
        try:
            jobs.append(request_queue.get(timeout=remaining))
        except queue.Empty:
            break
    return jobs


def _process_jobs(jobs: list[InferenceJob]):
    grouped_jobs: dict[tuple[Any, ...], list[InferenceJob]] = defaultdict(list)
    for job in jobs:
        grouped_jobs[_generation_key(job.request)].append(job)

    for group in grouped_jobs.values():
        try:
            responses = generate_batch_chat_completions([job.request for job in group])
            for job, response in zip(group, responses):
                _resolve_job(job, result=response)
        except Exception as exc:
            for job in group:
                _resolve_job(job, error=exc)


def batch_worker():
    while True:
        first_job = request_queue.get()
        jobs = _take_batch(first_job)
        _process_jobs(jobs)


batch_worker_thread = threading.Thread(target=batch_worker, daemon=True)
batch_worker_thread.start()


def stream_chat_completion(response: dict[str, Any]):
    content = response["choices"][0]["message"]["content"]
    chunk = {
        "id": response["id"],
        "object": "chat.completion.chunk",
        "created": response["created"],
        "model": response["model"],
        "choices": [
            {
                "index": 0,
                "delta": {"role": "assistant", "content": content},
                "finish_reason": None,
            }
        ],
    }
    yield f"data: {json.dumps(chunk, ensure_ascii=False)}\\n\\n"

    done_chunk = {
        "id": response["id"],
        "object": "chat.completion.chunk",
        "created": response["created"],
        "model": response["model"],
        "choices": [{"index": 0, "delta": {}, "finish_reason": "stop"}],
    }
    yield f"data: {json.dumps(done_chunk, ensure_ascii=False)}\\n\\n"
    yield "data: [DONE]\\n\\n"


@app.get("/health")
async def health():
    return {
        "status": "ok",
        "model": MODEL_NAME,
        "queue_size": request_queue.qsize(),
        "max_queue_size": MAX_QUEUE_SIZE,
        "max_batch_size": MAX_BATCH_SIZE,
    }


@app.get("/v1/models")
async def list_models():
    return {
        "object": "list",
        "data": [
            {
                "id": MODEL_NAME,
                "object": "model",
                "created": int(time.time()),
                "owned_by": "local-colab",
            }
        ],
    }


@app.post("/v1/chat/completions")
async def create_chat_completion(request: ChatCompletionRequest):
    if not request.messages:
        raise HTTPException(status_code=400, detail="messages 不能为空")
    loop = asyncio.get_running_loop()
    future = loop.create_future()
    job = InferenceJob(request=request, future=future, loop=loop)
    try:
        request_queue.put_nowait(job)
    except queue.Full:
        raise HTTPException(status_code=503, detail="服务器繁忙，请稍后再试。当前 GPU 推理队列已满。")

    try:
        response = await asyncio.wait_for(future, timeout=REQUEST_TIMEOUT_SECONDS)
    except asyncio.TimeoutError:
        raise HTTPException(status_code=504, detail="请求等待推理超时，请稍后重试。")
    if request.stream:
        return StreamingResponse(stream_chat_completion(response), media_type="text/event-stream")
    return response

In [5]:
import threading
import time

import nest_asyncio
import uvicorn
from google.colab import output

nest_asyncio.apply()

public_url = output.eval_js("google.colab.kernel.proxyPort(8000)")
print("Colab 代理访问地址:", public_url)
print("OpenAI base_url:", public_url + "/v1")

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)


server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)
print("服务已在后台启动")

INFO:     Started server process [7154]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Colab 代理访问地址: https://8000-gpu-l4-s-kkb-ass1a2-khqt0lleg37t-a.asia-southeast1-2.prod.colab.dev
OpenAI base_url: https://8000-gpu-l4-s-kkb-ass1a2-khqt0lleg37t-a.asia-southeast1-2.prod.colab.dev/v1
服务已在后台启动


In [7]:
# 发送 30 个不同的手写意图识别请求，并和预期标签对比。
# 这些样例是手写生成的，不从训练数据 jsonl 读取。
from concurrent.futures import ThreadPoolExecutor, as_completed
import openai
import time

client = openai.OpenAI(
    api_key="EMPTY",
    base_url="http://127.0.0.1:8000/v1",
)

INTENT_LABELS = [
    "查询物流", "催促配送", "修改订单", "取消订单", "申请退款", "退换货",
    "发票咨询", "支付咨询", "价格价保", "优惠活动", "商品咨询", "催促处理",
    "投诉反馈", "转人工", "问候确认", "业务咨询", "查询订单", "其他",
]

SYSTEM_PROMPT = (
    "你是电商客服意图识别助手。请根据用户当前话语判断主要意图。"
    "只能从以下标签中选择一个并只输出标签名："
    + "/".join(INTENT_LABELS)
)

INTENT_EVAL_CASES = [
    {"id": 1, "text": "我看物流三天没更新了，现在包裹到哪里了？", "expected_intent": "查询物流"},
    {"id": 2, "text": "能不能帮我催一下配送，今天必须收到。", "expected_intent": "催促配送"},
    {"id": 3, "text": "订单里的收货手机号写错了，可以帮我改一下吗？", "expected_intent": "修改订单"},
    {"id": 4, "text": "这单我不要了，麻烦现在取消。", "expected_intent": "取消订单"},
    {"id": 5, "text": "钱付了但东西不想要了，请给我退款。", "expected_intent": "申请退款"},
    {"id": 6, "text": "收到的鞋子小了一码，我想换大一号。", "expected_intent": "退换货"},
    {"id": 7, "text": "这个订单能开公司抬头的电子发票吗？", "expected_intent": "发票咨询"},
    {"id": 8, "text": "为什么我银行卡扣款了，订单还显示未支付？", "expected_intent": "支付咨询"},
    {"id": 9, "text": "我刚买完就降价了，可以申请价保吗？", "expected_intent": "价格价保"},
    {"id": 10, "text": "618 还有满减券可以领吗？", "expected_intent": "优惠活动"},
    {"id": 11, "text": "这款耳机支持主动降噪和蓝牙 5.3 吗？", "expected_intent": "商品咨询"},
    {"id": 12, "text": "售后审核卡了两天了，麻烦尽快处理。", "expected_intent": "催促处理"},
    {"id": 13, "text": "客服一直不回复，我要投诉这次服务。", "expected_intent": "投诉反馈"},
    {"id": 14, "text": "别自动回复了，给我转人工客服。", "expected_intent": "转人工"},
    {"id": 15, "text": "你好，请问有人在吗？", "expected_intent": "问候确认"},
    {"id": 16, "text": "你们门店周末营业到几点？", "expected_intent": "业务咨询"},
    {"id": 17, "text": "帮我查一下 20260621001 这个订单状态。", "expected_intent": "查询订单"},
    {"id": 18, "text": "今天北京天气怎么样？", "expected_intent": "其他"},
    {"id": 19, "text": "快递显示已揽收后就没动静了，帮我看看。", "expected_intent": "查询物流"},
    {"id": 20, "text": "配送员一直没送来，麻烦催派一下。", "expected_intent": "催促配送"},
    {"id": 21, "text": "我想把收货地址从上海改到杭州。", "expected_intent": "修改订单"},
    {"id": 22, "text": "刚下的订单还没发货，帮我取消掉。", "expected_intent": "取消订单"},
    {"id": 23, "text": "商品破损了，我要申请退款。", "expected_intent": "申请退款"},
    {"id": 24, "text": "衣服颜色发错了，我要退回去重新换。", "expected_intent": "退换货"},
    {"id": 25, "text": "支付时提示风控失败，怎么解决？", "expected_intent": "支付咨询"},
    {"id": 26, "text": "买二送一活动什么时候结束？", "expected_intent": "优惠活动"},
    {"id": 27, "text": "这个保温杯内胆是不锈钢的吗？", "expected_intent": "商品咨询"},
    {"id": 28, "text": "我的退款申请一直没人审核，帮我加急。", "expected_intent": "催促处理"},
    {"id": 29, "text": "你们这个处理结果我不认可，我要反馈问题。", "expected_intent": "投诉反馈"},
    {"id": 30, "text": "我想问下会员等级规则是怎么算的？", "expected_intent": "业务咨询"},
]


def normalize_intent(response_text: str) -> str:
    text = response_text.strip()
    for intent in INTENT_LABELS:
        if text == intent or intent in text:
            return intent
    return "其他"


def call_intent_api(case: dict) -> dict:
    started_at = time.time()
    resp = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": case["text"]},
        ],
        temperature=0.0,
        top_p=1.0,
        max_tokens=12,
    )
    raw_output = resp.choices[0].message.content.strip()
    pred_intent = normalize_intent(raw_output)
    return {
        **case,
        "raw_output": raw_output,
        "pred_intent": pred_intent,
        "is_correct": pred_intent == case["expected_intent"],
        "latency_seconds": round(time.time() - started_at, 3),
    }


results = []
with ThreadPoolExecutor(max_workers=30) as executor:
    futures = [executor.submit(call_intent_api, case) for case in INTENT_EVAL_CASES]
    for future in as_completed(futures):
        results.append(future.result())

results.sort(key=lambda item: item["id"])
correct_count = sum(1 for item in results if item["is_correct"])
accuracy = correct_count / len(results)

print("================== 30 条手写意图识别并发测试结果 ==================")
for item in results:
    mark = "正确" if item["is_correct"] else "错误"
    print(
        f"{item['id']:02d} | {mark} | 预期: {item['expected_intent']} | "
        f"预测: {item['pred_intent']} | 耗时: {item['latency_seconds']}s | "
        f"问题: {item['text']} | 原始输出: {item['raw_output']}"
    )

print("================================================================")
print(f"正确数: {correct_count}/{len(results)}")
print(f"准确率: {accuracy:.2%}")


INFO:     127.0.0.1:51122 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO:     127.0.0.1:51142 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO:     127.0.0.1:51148 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO:     127.0.0.1:51132 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO:     127.0.0.1:51164 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO:     127.0.0.1:51168 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO:     127.0.0.1:51174 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO:     127.0.0.1:51180 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO:     127.0.0.1:51166 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO:     127.0.0.1:51194 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO:     127.0.0.1:51224 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO:     127.0.0.1:51214 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO:     127.0.0.1:51198 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO:     127.0.0.1:51238 - "POST /v1/chat/completions HTTP/1.1"